# 02 — Tiền xử lý & Feature Engineering

**Pipeline tiền xử lý:**
1. Xử lý missing (median / mode)
2. Nhị phân hóa target (0 vs 1–4)
3. Mã hóa biến phân loại (Label Encoding)
4. Tách train/test (80/20, stratified)
5. Chuẩn hóa biến số (StandardScaler)
6. Cân bằng lớp (SMOTE)

**Feature Engineering:**
- Tạo đặc trưng phái sinh (age_group, interaction, ratio)
- Đánh giá Mutual Information
- Chọn top-K đặc trưng (ANOVA F-value)

In [1]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.dirname(os.getcwd()))

from src import load_params
from src.data.loader import load_raw_data
from src.data.cleaner import run_cleaning_pipeline
from src.features.builder import create_features, feature_importance

params = load_params()

## 2.1 Đọc dữ liệu gốc

In [2]:
df_raw = load_raw_data(params)
df_raw.head()

[LOADER] Đọc thành công: 920 dòng × 16 cột


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


## 2.2 Chạy pipeline tiền xử lý

Missing → Binarize target → Encode → Split → Scale → SMOTE  
Kết quả lưu vào `data/processed/`

> **Lưu ý anti-leakage:** Scale chỉ fit trên train, transform test riêng.

In [3]:
prep = run_cleaning_pipeline(df_raw, params)
print(f"\nX_train: {prep['X_train'].shape}, X_test: {prep['X_test'].shape}")
print(f"X_train_bal (SMOTE): {prep['X_train_bal'].shape}")


TIỀN XỬ LÝ DỮ LIỆU
[CLEANER] Xử lý missing xong. Còn thiếu: 0
[CLEANER] Nhị phân hóa target: {1: 509, 0: 411}
[CLEANER] Mã hóa 7 biến phân loại
[CLEANER] Tách train/test: Train=736, Test=184
  Train: {1: 407, 0: 329}
  Test:  {1: 102, 0: 82}
[CLEANER] Chuẩn hóa 6 biến số (StandardScaler)
[CLEANER] Chuẩn hóa 6 biến số (StandardScaler)
[CLEANER] SMOTE: 736 → 814 mẫu
  Phân bố sau: {1: 407, 0: 407}
[CLEANER] Đã lưu dữ liệu processed → c:\KHMT\BTL-DATAMINING\data/processed

X_train: (736, 13), X_test: (184, 13)
X_train_bal (SMOTE): (814, 13)


## 2.2b Thống kê trước – sau tiền xử lý

In [4]:
import pandas as pd

print("=" * 60)
print("THỐNG KÊ TRƯỚC – SAU TIỀN XỬ LÝ")
print("=" * 60)

# Missing trước vs sau
print("\n1) MISSING VALUES:")
print(f"   Trước: {df_raw.isnull().sum().sum()} giá trị thiếu "
      f"({df_raw.isnull().sum().sum()/(df_raw.shape[0]*df_raw.shape[1])*100:.2f}%)")
print(f"   Sau:   {prep['df_clean'].isnull().sum().sum()} giá trị thiếu")

# Phân bố target trước vs sau nhị phân hóa
print(f"\n2) TARGET DISTRIBUTION:")
print(f"   Gốc (0-4): {df_raw['num'].value_counts().sort_index().to_dict()}")
binary_dist = prep['df_clean']['num'].value_counts().sort_index().to_dict()
print(f"   Nhị phân:  {binary_dist}")
total = sum(binary_dist.values())
print(f"   Imbalance ratio: {binary_dist.get(0,0)/total*100:.1f}% / {binary_dist.get(1,0)/total*100:.1f}%")

# Kích thước trước/sau SMOTE
print(f"\n3) SMOTE RESAMPLING:")
print(f"   Train (trước SMOTE): {prep['X_train'].shape[0]} mẫu — {prep['y_train'].value_counts().to_dict()}")
print(f"   Train (sau SMOTE):   {prep['X_train_bal'].shape[0]} mẫu — {pd.Series(prep['y_train_bal']).value_counts().to_dict()}")
print(f"   Test (không đổi):    {prep['X_test'].shape[0]} mẫu — {prep['y_test'].value_counts().to_dict()}")

# Scale trước/sau
print(f"\n4) SCALING (StandardScaler):")
print(f"   X_train mean ≈ {prep['X_train'].mean().mean():.4f}  (kỳ vọng ≈ 0)")
print(f"   X_train std  ≈ {prep['X_train'].std().mean():.4f}  (kỳ vọng ≈ 1)")

print(f"\n{'='*60}")

THỐNG KÊ TRƯỚC – SAU TIỀN XỬ LÝ

1) MISSING VALUES:
   Trước: 1759 giá trị thiếu (11.95%)
   Sau:   0 giá trị thiếu

2) TARGET DISTRIBUTION:
   Gốc (0-4): {0: 411, 1: 265, 2: 109, 3: 107, 4: 28}
   Nhị phân:  {0: 411, 1: 509}
   Imbalance ratio: 44.7% / 55.3%

3) SMOTE RESAMPLING:
   Train (trước SMOTE): 736 mẫu — {1: 407, 0: 329}
   Train (sau SMOTE):   814 mẫu — {1: 407, 0: 407}
   Test (không đổi):    184 mẫu — {1: 102, 0: 82}

4) SCALING (StandardScaler):
   X_train mean ≈ 0.4165  (kỳ vọng ≈ 0)
   X_train std  ≈ 0.7586  (kỳ vọng ≈ 1)



### Nhận xét thống kê trước – sau

1. **Missing values:** Dữ liệu gốc có ~12% giá trị thiếu (chủ yếu `ca` 66%, `thal` 53%). Sau impute (median/mode) → 0% thiếu.
2. **Target:** Từ 5 mức (0–4) → nhị phân 0/1. Tỷ lệ ~45/55 — mất cân bằng nhẹ nhưng cần xử lý vì trong y tế, bỏ sót bệnh nhân (FN) rất nguy hiểm.
3. **SMOTE:** Cân bằng hoàn toàn 50/50 trên tập train. Tập test **không** SMOTE → đánh giá trung thực.
4. **Scaling:** Mean ≈ 0, Std ≈ 1 → StandardScaler hoạt động đúng. Scaler chỉ fit trên train → **không data leakage**.

## 2.3 Feature Importance (Mutual Information)

In [5]:
fi = feature_importance(prep["X_train"], prep["y_train"])
fi

[FEATURE] Tầm quan trọng (MI):
Đặc trưng  MI Score
       cp  0.117313
    exang  0.101604
   thalch  0.081064
     chol  0.073390
  oldpeak  0.066868
      sex  0.055842
      age  0.042506
     thal  0.035945
 trestbps  0.019987
       ca  0.015228
  restecg  0.000000
      fbs  0.000000
    slope  0.000000


,Đặc trưng,MI Score
0,cp,0.117313
1,exang,0.101604
2,thalch,0.081064
3,chol,0.073390
4,oldpeak,0.066868
5,sex,0.055842
6,age,0.042506
7,thal,0.035945
8,trestbps,0.019987
9,ca,0.015228


In [6]:
df_feat = create_features(prep["df_encoded"], params)
print(f"Cột sau khi tạo đặc trưng: {df_feat.shape[1]}")
df_feat.head()

[FEATURE] Tạo đặc trưng mới: 21 cột tổng cộng
Cột sau khi tạo đặc trưng: 21


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,...,oldpeak,slope,ca,thal,num,age_group,age_bp_interaction,chol_age_ratio,hr_oldpeak_diff,bp_hr_ratio
0,1,63.0,1,Cleveland,3,145.0,233.0,1,0,150.0,...,2.3,0,0.0,0,0,3.0,9135.0,3.640625,127.0,0.960265
1,2,67.0,1,Cleveland,0,160.0,286.0,0,0,108.0,...,1.5,1,3.0,1,1,3.0,10720.0,4.205882,93.0,1.467890
2,3,67.0,1,Cleveland,0,120.0,229.0,0,0,129.0,...,2.6,1,2.0,2,1,3.0,8040.0,3.367647,103.0,0.923077
3,4,37.0,1,Cleveland,2,130.0,250.0,0,1,187.0,...,3.5,0,0.0,1,0,0.0,4810.0,6.578947,152.0,0.691489
4,5,41.0,0,Cleveland,1,130.0,204.0,0,0,172.0,...,1.4,2,0.0,1,0,1.0,5330.0,4.857143,158.0,0.751445


### Nhận xét Feature Engineering

**Mutual Information (MI) — Top đặc trưng:**
- `cp` (loại đau ngực): MI cao nhất → **biến quan trọng nhất** để phân biệt bệnh/không bệnh
- `exang` (đau ngực gắng sức): MI cao thứ 2 → triệu chứng lâm sàng mạnh
- `thalch`, `chol`, `oldpeak`: nhóm biến số có MI đáng kể → đóng góp cho mô hình
- `fbs`, `restecg`, `slope`: MI ≈ 0 → ít đóng góp, nhưng giữ lại vì ensemble model có thể khai thác

**Đặc trưng phái sinh:**
- `age_group`: rời rạc hóa tuổi → giúp cây quyết định + luật kết hợp
- Interaction terms (cp × exang, age × trestbps): bắt non-linearity giữa triệu chứng
- Ratios (chol/age, oldpeak/thalch): chuẩn hóa theo sinh lý bệnh nhân

→ Tổng cộng 21 đặc trưng (13 gốc + 8 phái sinh) sẵn sàng cho modeling.